In [1]:
import pandas as pd
import numpy as np
import os

In [4]:
DATA_DIR = "/Users/dhairyabhatt/code_dir/Data analytics projects/world_health/data/DrinkingWater"
def extract_metadata(filepath):
    file_name = os.path.basename(filepath)
    print(f"FILE:{file_name}")

    df = pd.read_csv(filepath,low_memory=False)
    print(f"\nshape: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(f"\n columns and data types")
    print(df.dtypes.to_string())

    print(f"\n Missing values")
    nulls = df.isnull().sum()
    nulls = nulls[nulls>0]
    if nulls.empty:
        print('None')
    else:
        pct = (nulls/len(df)*100).round(1)
        print(pd.DataFrame({"Missing":nulls,"percentage":pct}).to_string())

    #categorical Columns

    categorical_cols = [c for c in df.columns 
                        if c not in ('OBS_VALUE','TIME_PERIOD','obs_value','time_period') 
                        and df[c].dtype == 'object'
                        ]
    
    #printing unique values per categorical column

    print("Unique values per category column")
    for col in categorical_cols:
        unique_vals = df[col].dropna().unique()
        n = len(unique_vals)
        sample = sorted(unique_vals)[:10]
        print(f"{col}({n}unique,sample): {sample}")

#doing time analysis
    time_vals = next((c for c in df.columns if c.upper == "TIME_PERIOD"),None)
    if time_vals:
        times = df[time_vals].dropna().sort_values()
        print("Time range")
        print(f"From : {times.iloc[0]}")
        print(f"To : {times.iloc[-1]}")
        print(f"Count: {times.nunique()}")

# OBS_VALUE quick stats
    val_col = next((c for c in df.columns if c.upper() == "OBS_VALUE"), None)
    if val_col:
        df[val_col] = pd.to_numeric(df[val_col], errors="coerce")
        print(f"\n--- OBS_VALUE Stats ---")
        print(df[val_col].describe().round(4).to_string())

csv_files = [f for f in os.listdir(DATA_DIR) if f.endswith(".csv")]

if not csv_files:
    print(f"No CSV files found in {DATA_DIR}")
else:
    print(f"Found {len(csv_files)} CSV files: {csv_files}")
    for fname in csv_files:
        extract_metadata(os.path.join(DATA_DIR, fname))

    

Found 4 CSV files: ['basicDrinkingWaterServices.csv', 'safelySanitization.csv', 'basicHandWashing.csv', 'atLeastBasicSanitizationServices.csv']
FILE:basicDrinkingWaterServices.csv

shape: 3,455 rows x 4 columns

 columns and data types
Location             str
Period             int64
Indicator            str
First Tooltip    float64

 Missing values
None
Unique values per category column
FILE:safelySanitization.csv

shape: 3,655 rows x 5 columns

 columns and data types
Location             str
Indicator            str
Period             int64
Dim1                 str
First Tooltip    float64

 Missing values
None
Unique values per category column
FILE:basicHandWashing.csv

shape: 2,726 rows x 5 columns

 columns and data types
Location             str
Indicator            str
Period             int64
Dim1                 str
First Tooltip    float64

 Missing values
None
Unique values per category column
FILE:atLeastBasicSanitizationServices.csv

shape: 9,368 rows x 5 columns

 colum

In [3]:
data_dir = "/Users/dhairyabhatt/code_dir/Data analytics projects/world_bank/data"
    

In [10]:
#printing the main parent directory and the sub directories
parent_data_dir = "/Users/dhairyabhatt/code_dir/Data analytics projects/world_health/data"
sub_dirs = os.listdir(parent_data_dir)[1:]
print(sub_dirs)

['CommunicableDisease', 'HealthWorkForce', 'LifeExpectencyAndHealthyLifeExpectency', 'DrinkingWater', 'UHCincludingFinancialRisk']


In [37]:
path_dict = {}
from pathlib import Path
for i in sub_dirs:
    absoulte_path = os.path.abspath(i)
    print(absoulte_path)
    path_dict[i] = absoulte_path

#the path_dict contains the absolute path of every subfolder in the main data dir

/Users/dhairyabhatt/code_dir/Data analytics projects/world_health/notebooks/CommunicableDisease
/Users/dhairyabhatt/code_dir/Data analytics projects/world_health/notebooks/HealthWorkForce
/Users/dhairyabhatt/code_dir/Data analytics projects/world_health/notebooks/LifeExpectencyAndHealthyLifeExpectency
/Users/dhairyabhatt/code_dir/Data analytics projects/world_health/notebooks/DrinkingWater
/Users/dhairyabhatt/code_dir/Data analytics projects/world_health/notebooks/UHCincludingFinancialRisk


In [21]:
os.listdir("/Users/dhairyabhatt/code_dir/Data analytics projects/world_health/data/CommunicableDisease")

['hepatitusBsurfaceAntigen.csv',
 'incedenceOfMalaria.csv',
 'incedenceOfTuberculosis.csv',
 'newHivInfections.csv',
 'interventionAgianstNTDs.csv']

In [43]:
def create_dict(path):
    #get the possible dirs
    files = os.listdir(path)
    dict_name = {}
    for i in files:
        abs_path = os.path.abspath(i)
        dict_name[i] = abs_path
    return dict_name


CommunicableDieases = create_dict("/Users/dhairyabhatt/code_dir/Data analytics projects/world_health/data/CommunicableDisease")
DrinkingWater = create_dict("/Users/dhairyabhatt/code_dir/Data analytics projects/world_health/data/DrinkingWater")
HealthWorkForce = create_dict("/Users/dhairyabhatt/code_dir/Data analytics projects/world_health/data/HealthWorkForce")
LifeExpectancyAndHealthLifeExpectancy = create_dict("/Users/dhairyabhatt/code_dir/Data analytics projects/world_health/data/CommunicableDisease")
UHCincludingFinancialRisk = create_dict("/Users/dhairyabhatt/code_dir/Data analytics projects/world_health/data/CommunicableDisease")
    


In [54]:
import os
import pandas as pd
from pathlib import Path


def get_fill_pct(series: pd.Series) -> str:
    pct = (series.notna().sum() / len(series)) * 100 if len(series) > 0 else 0
    return f"{pct:.1f}%"


def get_notes(series: pd.Series) -> str:
    if pd.api.types.is_numeric_dtype(series):
        lo, hi = series.min(), series.max()
        return f"range: {lo} – {hi}"
    unique = series.nunique(dropna=True)
    return f"{unique} unique values"


def file_overview_md(filepath: Path, df: pd.DataFrame) -> str:
    size_kb = filepath.stat().st_size / 1024
    rows = [
        ("File name",   filepath.name),
        ("Folder",      filepath.parent.name),
        ("Path",        str(filepath)),
        ("File size",   f"{size_kb:.1f} KB"),
        ("Row count",   len(df)),
        ("Column count",len(df.columns)),
    ]
    lines = ["## File Overview", "| Property | Value |", "| --- | --- |"]
    for prop, val in rows:
        lines.append(f"| {prop} | {val} |")
    return "\n".join(lines)


def column_metadata_md(df: pd.DataFrame) -> str:
    lines = [
        "## Column Metadata",
        "### Identity & Location",
        "| Column | Type | Fill % | Notes |",
        "| --- | --- | --- | --- |",
    ]
    for col in df.columns:
        series = df[col]
        lines.append(f"| {col} | {series.dtype} | {get_fill_pct(series)} | {get_notes(series)} |")
    return "\n".join(lines)


def extract_metadata(data_dir: str, output_file: str = "metadata_report.md") -> None:
    data_path = Path(data_dir)
    csv_files = sorted(data_path.glob("*/*.csv"))  # one level deep: Data/folder/*.csv

    if not csv_files:
        print(f"No CSV files found under {data_path}")
        return

    sections = []
    for filepath in csv_files:
        print(f"Processing: {filepath}")
        try:
            df = pd.read_csv(filepath)
            section = "\n\n".join([
                f"# {filepath.name}",
                file_overview_md(filepath, df),
                column_metadata_md(df),
            ])
            sections.append(section)
        except Exception as e:
            sections.append(f"# {filepath.name}\n\n> ⚠️ Could not read file: {e}")

    report = "\n\n---\n\n".join(sections)
    Path(output_file).write_text(report, encoding="utf-8")
    print(f"\nDone. Report written to: {output_file} ({len(csv_files)} files processed)")


if __name__ == "__main__":
    extract_metadata("/Users/dhairyabhatt/code_dir/Data analytics projects/world_health/data")

Processing: /Users/dhairyabhatt/code_dir/Data analytics projects/world_health/data/CommunicableDisease/hepatitusBsurfaceAntigen.csv
Processing: /Users/dhairyabhatt/code_dir/Data analytics projects/world_health/data/CommunicableDisease/incedenceOfMalaria.csv
Processing: /Users/dhairyabhatt/code_dir/Data analytics projects/world_health/data/CommunicableDisease/incedenceOfTuberculosis.csv
Processing: /Users/dhairyabhatt/code_dir/Data analytics projects/world_health/data/CommunicableDisease/interventionAgianstNTDs.csv
Processing: /Users/dhairyabhatt/code_dir/Data analytics projects/world_health/data/CommunicableDisease/newHivInfections.csv
Processing: /Users/dhairyabhatt/code_dir/Data analytics projects/world_health/data/DrinkingWater/atLeastBasicSanitizationServices.csv
Processing: /Users/dhairyabhatt/code_dir/Data analytics projects/world_health/data/DrinkingWater/basicDrinkingWaterServices.csv
Processing: /Users/dhairyabhatt/code_dir/Data analytics projects/world_health/data/DrinkingWat